# EEG Trigger Press · LightGBM Pipeline Parameter Optimisation

Companion to `eeg_lgb_focused.ipynb`.  
Tree hyper-parameters are **fixed** (loaded from `models/lgb_demo.pkl` or defaults).  
Optuna searches the **pipeline** parameters:

| Parameter | What it controls |
|---|---|
| `T` (window ms) | How much EEG history feeds the model |
| `HORIZON` (ms) | How far ahead the label looks for a press |
| `PERSIST_K` | Consecutive frames above threshold before firing |

**Note**: `REFRACTORY` is a fixed deployment parameter — see config cell. Searching it lets Optuna exploit large values to inflate FA/min suppression without improving the model (refractory only filters FA *counting* in the metric, not event sensitivity).

**Objective**: maximise `event_sensitivity / (1 + α · FA/min)` via LOTO cross-validation.  
Bounded `[0, 1]`: equals sensitivity when FA=0, halves it when `FA/min = 1/α`.


In [1]:
import warnings

warnings.filterwarnings("ignore")
import os
import sys

import lightgbm as lgb
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
from scipy.signal import welch
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)

sys.path.insert(0, "..")
from src.postproc import event_detection_metrics, persistence_gate
from src.preprocessing import (
    build_windows,
    euclidean_align,
    load_trial,
    make_horizon_labels,
    make_post_press_mask,
)

optuna.logging.set_verbosity(optuna.logging.WARNING)

DARK, CARD, EDGE = "#0a0e17", "#111827", "#1f2937"
TEAL, CORAL, GOLD, WHITE = "#00e5cc", "#ff4f5e", "#ffc947", "#f0f4ff"
LIME, PURPLE = "#a8ff3e", "#c084fc"

print("Imports OK  (LightGBM", lgb.__version__, ")")

Imports OK  (LightGBM 4.6.0 )


## 1 · Config

In [2]:
SUBJECT = 16
TACHE = "gng"
ALL_TRIALS = list(range(10))
TEST_TRIALS = list(range(10))
DATA_PATH = f"../data/subject{SUBJECT}/"

# ── Auto-detect sampling rate ──────────────────────────────────────────────
_first = f"../data/subject{SUBJECT}/subject_{SUBJECT}_tache_{TACHE}_trial_0.csv"
_ts = pd.read_csv(_first, sep=None, engine="python", usecols=["timestamp"])[
    "timestamp"
].values
FS_RAW = round(len(_ts) / (_ts[-1] - _ts[0]))
DECIM = 4 if FS_RAW >= 100 else 1
FS = FS_RAW
FS_EFF = FS // DECIM
print(f"Detected FS={FS_RAW} Hz  DECIM={DECIM}  FS_EFF={FS_EFF} Hz")

LP, HP = 0.5, 28.0
TARGET = "button"
USE_EA = False
DESPIKE_SIGMA = 5.0
SKIP_SECONDS = 20.0

# ── Pipeline search space (milliseconds) ────────────────────────────────────
T_MS_MIN, T_MS_MAX = 200, 2500
HORIZON_MS_MIN, HORIZON_MS_MAX = 500, 2500
PERSIST_K_MIN, PERSIST_K_MAX = 1, 20

# ── Fixed deployment parameter (NOT searched by Optuna) ─────────────────────
# Refractory only suppresses FA *counting* in event_detection_metrics; it does
# NOT affect event sensitivity. Searching it lets Optuna push the value to its
# maximum to inflate the objective without improving predictions. Set below the
# 5th-percentile inter-press interval (~6-9 s mean IPI for this dataset).
REFRACTORY_MS = 1000

# Upper bound for window precomputation
T_MAX = int(T_MS_MAX / 1000 * FS_EFF)
T_MIN = int(T_MS_MIN / 1000 * FS_EFF)

# ── Objective ──────────────────────────────────────────────────────────────
# Maximise:  event_sensitivity / (1 + OBJ_ALPHA * fa_per_min)
# Bounded [0, 1].  FA/min = 1/OBJ_ALPHA halves the score.
# OBJ_ALPHA=0.1 → FA/min=10 cuts score in half.
OBJ_ALPHA = 0.1

# ── Fixed misc ──────────────────────────────────────────────────────────────
MIN_SPEC = 0.95
SMOOTH_WIN = max(1, int(0.1 * FS_EFF))  # 100 ms
POST_PRESS_BLANK_MS = 1500
N_OPTUNA = 15
LGB_JOBS = -1
TRAIN_CAP = 1_000_000
SEED = 42

print(f"FS_EFF={FS_EFF} Hz  T_MAX={T_MAX} samp ({T_MAX/FS_EFF:.1f} s)")
print(f"T search      : {T_MS_MIN}-{T_MS_MAX} ms")
print(f"HORIZON search: {HORIZON_MS_MIN}-{HORIZON_MS_MAX} ms")
print(f"PERSIST_K     : {PERSIST_K_MIN}-{PERSIST_K_MAX} frames")
print(f"REFRACTORY    : {REFRACTORY_MS} ms  [fixed — not searched]")
print(
    f"Objective     : sens / (1 + {OBJ_ALPHA} * FA/min)  in [0,1]  (direction=maximize)"
)
print(f"USE_EA={USE_EA}  N_OPTUNA={N_OPTUNA}")

Detected FS=243 Hz  DECIM=4  FS_EFF=60 Hz
FS_EFF=60 Hz  T_MAX=150 samp (2.5 s)
T search      : 200-2500 ms
HORIZON search: 500-2500 ms
PERSIST_K     : 1-20 frames
REFRACTORY    : 1000 ms  [fixed — not searched]
Objective     : sens / (1 + 0.1 * FA/min)  in [0,1]  (direction=maximize)
USE_EA=False  N_OPTUNA=15


## 2 · Load & Filter

Horizon labels are **not** applied here — HORIZON is a search variable and will be
recomputed inside each Optuna trial via `make_horizon_labels`.


In [3]:
print("Loading (CAR + trial_zscore + 0.5-28 Hz) ...")
raw_data, EEG_COLS = {}, None
led_data = {}

for t in ALL_TRIALS:
    df, cols = load_trial(
        SUBJECT,
        TACHE,
        t,
        DATA_PATH,
        eeg_cols=EEG_COLS,
        fs=FS,
        lp=LP,
        hp=HP,
        decimate=DECIM,
        car=True,
        trial_zscore=True,
        clip_percentile=1.0,
        despike_sigma=DESPIKE_SIGMA,
        skip_seconds=SKIP_SECONDS,
    )
    if EEG_COLS is None:
        EEG_COLS = cols
    X_t = df[EEG_COLS].values.astype(np.float32)
    y_mom_t = df[TARGET].values.astype(np.int8)
    raw_data[t] = (X_t, y_mom_t)  # no horizon labels
    led_data[t] = df["led"].values.astype(np.int8) if "led" in df.columns else None
    n_pos = int(y_mom_t.sum())
    print(
        f"  trial {t:2d}: {len(df):5,} samp  press={n_pos:4,} ({n_pos/len(df)*100:.1f}%)"
    )

N_CH = len(EEG_COLS)
print(f"\n{N_CH} channels: {EEG_COLS}")

Loading (CAR + trial_zscore + 0.5-28 Hz) ...
  trial  0: 10,749 samp  press=  30 (0.3%)
  trial  1: 12,458 samp  press=  30 (0.2%)
  trial  2: 10,431 samp  press=  30 (0.3%)
  trial  3: 11,167 samp  press=  28 (0.3%)
  trial  4: 11,047 samp  press=  30 (0.3%)
  trial  5: 11,342 samp  press=  30 (0.3%)
  trial  6: 11,022 samp  press=  30 (0.3%)
  trial  7: 11,214 samp  press=  30 (0.3%)
  trial  8: 10,873 samp  press=  29 (0.3%)
  trial  9: 10,696 samp  press=  30 (0.3%)

16 channels: ['ch1', 'ch2', 'ch3', 'ch4', 'ch5', 'ch6', 'ch7', 'ch8', 'ch9', 'ch10', 'ch11', 'ch12', 'ch13', 'ch14', 'ch15', 'ch16']


## 3 · Trial Quality Check

In [4]:
from src.preprocessing import trial_quality

SKIP_BAD_TRIALS = True

print(
    f"{'Trial':>6}  {'Presses':>7}  {'Dur(s)':>6}  {'ArtFrac':>8}  {'MaxKurt':>8}  {'Flag':>5}  Reasons"
)
print("-" * 76)
bad_trials = []
for t in list(ALL_TRIALS):
    X_t, y_mom_t = raw_data[t]
    q = trial_quality(X_t, y_mom_t, FS_EFF)
    if q["flat_channels"]:
        X_t[:, q["flat_channels"]] = 0.0
        raw_data[t] = (X_t, y_mom_t)
    marker = {"ok": "   ", "warn": "W  ", "bad": "X  "}[q["flag"]]
    print(
        f"  {t:4d}  {q['n_presses']:>7}  {q['duration_s']:>6.1f}  "
        f"{q['artifact_frac']:>8.4f}  {q['max_kurtosis']:>8.2f}  "
        f"{marker}{q['flag']:>4}  {'; '.join(q['reasons'])}"
    )
    if q["flag"] == "bad":
        bad_trials.append(t)

if SKIP_BAD_TRIALS and bad_trials:
    for t in bad_trials:
        del raw_data[t]
        if t in led_data:
            del led_data[t]
    ALL_TRIALS = [t for t in ALL_TRIALS if t not in bad_trials]
    TEST_TRIALS = [t for t in TEST_TRIALS if t not in bad_trials]
    print(f"\nDropped {len(bad_trials)} bad trial(s): {bad_trials}")
else:
    print(f"\nAll {len(ALL_TRIALS)} trials retained.")

 Trial  Presses  Dur(s)   ArtFrac   MaxKurt   Flag  Reasons
----------------------------------------------------------------------------
     0       30   179.2    0.0026     51.02  W  warn  max_kurt=51.0 > 10.0
     1       30   207.6    0.0224     29.57  W  warn  max_kurt=29.6 > 10.0
     2       30   173.8    0.0179     28.58  W  warn  max_kurt=28.6 > 10.0
     3       28   186.1    0.0059     20.05  W  warn  max_kurt=20.1 > 10.0
     4       30   184.1    0.0244     18.89  W  warn  max_kurt=18.9 > 10.0
     5       30   189.0    0.0000     76.21  W  warn  max_kurt=76.2 > 10.0
     6       30   183.7    0.0182     33.65  W  warn  max_kurt=33.7 > 10.0
     7       30   186.9    0.0304     18.29  W  warn  max_kurt=18.3 > 10.0
     8       29   181.2    0.0238     14.84  W  warn  max_kurt=14.8 > 10.0
     9       30   178.3    0.0292     19.39  W  warn  max_kurt=19.4 > 10.0

All 10 trials retained.


## 4 · Feature Engineering

In [5]:
def temporal_features(X3d: np.ndarray, n_segs: int = 4) -> np.ndarray:
    N, T, C = X3d.shape
    X64 = X3d.astype(np.float64)
    t_c = np.arange(T, dtype=np.float64) - (T - 1) / 2.0
    t_var = (t_c**2).sum() + 1e-12
    mean_ = X64.mean(axis=1)
    std_ = X64.std(axis=1)
    slope_ = (X64 * t_c[None, :, None]).sum(axis=1) / t_var
    min_ = X64.min(axis=1)
    argm_ = X64.argmin(axis=1) / T
    rms_ = np.sqrt((X64**2).mean(axis=1))
    base = np.stack([mean_, std_, slope_, min_, argm_, rms_], axis=2)
    seg_size = max(1, T // n_segs)
    segs = np.stack(
        [
            X64[:, i * seg_size : (i + 1) * seg_size, :].mean(axis=1)
            for i in range(n_segs)
        ],
        axis=2,
    )
    return (
        np.concatenate([base, segs], axis=2)
        .reshape(N, C * (6 + n_segs))
        .astype(np.float32)
    )


def freq_features(X3d: np.ndarray, fs: float, batch: int = 2000) -> np.ndarray:
    N, T, C = X3d.shape
    nperseg = min(T, 64)
    out = np.zeros((N, C * 5), dtype=np.float32)
    for s in range(0, N, batch):
        xb = X3d[s : s + batch].transpose(0, 2, 1)
        f, p = welch(xb, fs=fs, nperseg=nperseg, axis=-1)
        df = f[1] - f[0]
        dp = p[:, :, (f >= 0.5)  & (f <= 4.0) ].sum(-1) * df
        tp = p[:, :, (f >= 4.0)  & (f <= 8.0) ].sum(-1) * df
        al = p[:, :, (f >= 8.0)  & (f <= 13.0)].sum(-1) * df
        be = p[:, :, (f >= 13.0) & (f <= 28.0)].sum(-1) * df
        ma = f > 0
        pn = p[:, :, ma] / (p[:, :, ma].sum(-1, keepdims=True) + 1e-12)
        se = -(pn * np.log(pn + 1e-12)).sum(-1)
        out[s : s + batch, 0::5] = dp
        out[s : s + batch, 1::5] = tp
        out[s : s + batch, 2::5] = al
        out[s : s + batch, 3::5] = be
        out[s : s + batch, 4::5] = se
    return out


def hjorth_features(X3d: np.ndarray) -> np.ndarray:
    d1 = np.diff(X3d, axis=1)
    d2 = np.diff(d1, axis=1)
    v0 = X3d.var(axis=1) + 1e-12
    v1 = d1.var(axis=1) + 1e-12
    v2 = d2.var(axis=1) + 1e-12
    mob = np.sqrt(v1 / v0)
    cmp = np.sqrt(v2 / v1) / (mob + 1e-12)
    return np.stack([v0, mob, cmp], axis=2).reshape(len(X3d), -1).astype(np.float32)


def features_for_T(T: int, X_3d: np.ndarray = None) -> np.ndarray:
    src = X_3d if X_3d is not None else X_max
    X_t = src[:, -T:, :]
    return np.concatenate(
        [temporal_features(X_t), freq_features(X_t, FS_EFF), hjorth_features(X_t)],
        axis=1,
    ).astype(np.float32)


def mf_features(X3d: np.ndarray, template: np.ndarray) -> np.ndarray:
    t_norm = (template - template.mean(0)) / (template.std(0) + 1e-8)
    mu = X3d.mean(axis=1, keepdims=True)
    sd = X3d.std(axis=1, keepdims=True) + 1e-8
    return ((X3d - mu) / sd * t_norm[None]).mean(axis=1).astype(np.float32)


FEAT_NAMES = (
    [
        f"ch{c+1}_{f}"
        for c in range(N_CH)
        for f in [
            "mean",
            "std",
            "slope",
            "min",
            "argmin",
            "rms",
            "q1",
            "q2",
            "q3",
            "q4",
        ]
    ]
    + [f"ch{c+1}_{f}" for c in range(N_CH) for f in ["delta", "theta", "alpha", "beta", "entropy"]]
    + [
        f"ch{c+1}_{f}"
        for c in range(N_CH)
        for f in ["activity", "mobility", "complexity"]
    ]
    + [f"ch{c+1}_mf_corr" for c in range(N_CH)]
)
print(
    f"Feature names: {len(FEAT_NAMES)}  (temporal {N_CH*10} + freq {N_CH*5} + Hjorth {N_CH*3} + MF {N_CH})"
)

Feature names: 304  (temporal 160 + freq 80 + Hjorth 48 + MF 16)


## 5 · Pre-compute Windows

`X_max` stores the longest possible window per sample. `y_mom_raw_per_trial` stores the full raw label sequence per trial so horizon labels can be recomputed for any HORIZON inside Optuna.


In [6]:
print(f"Building windows at T_MAX={T_MAX} ({T_MAX/FS_EFF:.1f} s) ...")
_X, _ymom, _tid, _pp = [], [], [], []
y_mom_raw_per_trial = {}  # full pre-window y_mom per trial, for relabeling

for t in ALL_TRIALS:
    X_t, y_mom_t = raw_data[t]
    Xw, y_mom_w = build_windows(
        X_t, y_mom_t.astype(np.float32), T_MAX, per_window_norm=False
    )
    n = len(Xw)
    pp = make_post_press_mask(y_mom_t, int(POST_PRESS_BLANK_MS / 1000 * FS_EFF))
    pp_w = pp[T_MAX:][:n]
    _X.append(Xw[:n].reshape(-1, T_MAX, N_CH))
    _ymom.append(y_mom_w[:n].astype(np.int8))
    _tid.append(np.full(n, t, dtype=np.int32))
    _pp.append(pp_w[:n])
    y_mom_raw_per_trial[t] = y_mom_t  # full trial for make_horizon_labels

X_max = np.concatenate(_X)
y_mom_all = np.concatenate(_ymom)
tid_all = np.concatenate(_tid)
post_press_mask_all = np.concatenate(_pp)
del _X, _ymom, _tid, _pp

_sn = int((y_mom_all > 0).sum())
_sN = len(y_mom_all)
SPW = (_sN - _sn) / _sn
print(f"  {_sN:,} windows  {_sn:,} press frames ({_sn/_sN*100:.1f}%)  SPW={SPW:.1f}")
print(f"  X_max: {X_max.shape}")

Building windows at T_MAX=150 (2.5 s) ...
  109,499 windows  289 press frames (0.3%)  SPW=377.9
  X_max: (109499, 150, 16)


## 6 · Fixed Tree Parameters

Loaded from `models/lgb_demo.pkl` when available (run `eeg_lgb_focused → lgb-save` first).
Falls back to reasonable defaults otherwise.


In [7]:
import pickle

_pkl = "models/lgb_demo.pkl"
if os.path.exists(_pkl):
    with open(_pkl, "rb") as _fh:
        _art = pickle.load(_fh)
    _m = _art["model"]
    FIXED_PARAMS = dict(
        n_estimators=_m.n_estimators,
        learning_rate=_m.learning_rate,
        max_depth=_m.max_depth,
        num_leaves=_m.num_leaves,
        min_child_samples=_m.min_child_samples,
        subsample=_m.subsample,
        colsample_bytree=_m.colsample_bytree,
        reg_lambda=_m.reg_lambda,
    )
    print(f"Tree params loaded from {_pkl}:")
    del _art, _m
else:
    FIXED_PARAMS = dict(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        num_leaves=63,
        min_child_samples=30,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
    )
    print(f"{_pkl} not found — using defaults:")

for k, v in FIXED_PARAMS.items():
    print(f"  {k:22s}: {v}")

Tree params loaded from models/lgb_demo.pkl:
  n_estimators          : 508
  learning_rate         : 0.11069143219393454
  max_depth             : 8
  num_leaves            : 102
  min_child_samples     : 18
  subsample             : 0.6792328642721364
  colsample_bytree      : 0.5579345297625649
  reg_lambda            : 2.8340904295147746


## 7 · Utilities

In [8]:
def subsample(X, y, n, seed):
    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(len(y), min(n, len(y)), replace=False))
    return X[idx], y[idx]


def smooth(scores, window=SMOOTH_WIN):
    if window <= 1:
        return scores.copy()
    return (
        pd.Series(scores)
        .rolling(window, min_periods=1)
        .mean()
        .values.astype(np.float32)
    )


def thresh_at_spec(fpr, tpr, thresholds, min_spec=MIN_SPEC):
    for spec in [min_spec, 0.90, 0.85, 0.80, 0.70, 0.60]:
        valid = (1 - fpr) >= spec
        if valid.any() and tpr[valid].max() > 0:
            if spec < min_spec:
                print(f"    [thresh_at_spec] relaxed {min_spec:.0%} -> {spec:.0%}")
            return float(thresholds[np.argmax(tpr[valid])])
    return float(thresholds[np.argmax(tpr - fpr)])


def relabel_with_horizon(horizon: int) -> np.ndarray:
    """Recompute window-aligned horizon labels for every trial."""
    y_labeled = np.empty(len(y_mom_all), dtype=np.float32)
    for t in ALL_TRIALS:
        m = tid_all == t
        y_h = make_horizon_labels(y_mom_raw_per_trial[t], horizon)
        n = int(m.sum())
        y_labeled[m] = y_h[T_MAX:][:n]
    return y_labeled


def loto_eval(
    model_fn,
    X_3d,
    T_cur,
    y,
    tid,
    y_mom_arr,
    on_fit=None,
    use_mf=True,
    horizon=None,
    persist_k=1,
    refractory=None,
    post_press_mask=None,
):
    """Full LOTO CV. horizon and refractory must be passed explicitly."""
    if horizon is None or refractory is None:
        raise ValueError("Pass horizon and refractory explicitly.")

    scores = np.zeros(len(y), dtype=np.float32)
    fold_results = []

    for test_trial in TEST_TRIALS:
        te = tid == test_trial
        tr = ~te

        X_tr_3d = X_3d[tr, -T_cur:, :]
        X_te_3d = X_3d[te, -T_cur:, :]

        if USE_EA:
            X_tr_ea, W = euclidean_align(X_tr_3d)
            X_te_ea = (X_te_3d @ W.T).astype(np.float32)
        else:
            X_tr_ea = X_tr_3d.astype(np.float32)
            X_te_ea = X_te_3d.astype(np.float32)

        X_feat_tr = features_for_T(T_cur, X_tr_ea)
        X_feat_te = features_for_T(T_cur, X_te_ea)

        if use_mf:
            press_idx = np.where(y_mom_arr[tr] > 0)[0]
            template = (
                X_tr_ea[press_idx].mean(0)
                if len(press_idx) > 0
                else np.zeros((T_cur, X_tr_ea.shape[2]), dtype=np.float32)
            )
            X_feat_tr = np.concatenate(
                [X_feat_tr, mf_features(X_tr_ea, template)], axis=1
            )
            X_feat_te = np.concatenate(
                [X_feat_te, mf_features(X_te_ea, template)], axis=1
            )

        y_tr = y[tr]
        if post_press_mask is not None:
            keep = (y_tr == 1) | ~post_press_mask[tr]
            X_feat_tr, y_tr = X_feat_tr[keep], y_tr[keep]

        model = model_fn(X_feat_tr, y_tr)
        if on_fit is not None:
            on_fit(model)

        sc_tr = model.predict_proba(X_feat_tr)[:, 1]
        sc_te = model.predict_proba(X_feat_te)[:, 1]

        fpr_tr, tpr_tr, thr_tr = roc_curve(y_tr, sc_tr)
        thresh = thresh_at_spec(fpr_tr, tpr_tr, thr_tr)

        scores[te] = sc_te.astype(np.float32)
        auc = roc_auc_score(y[te], sc_te)
        ap = average_precision_score(y[te], sc_te)
        fold_results.append(dict(trial=test_trial, auc=auc, ap=ap, thresh=thresh))
        print(
            f"  fold {test_trial:2d}  AUC={auc:.4f}  AP={ap:.4f}  thresh={thresh:.3f}"
        )

    # Smooth per trial
    scores_s = np.zeros_like(scores)
    for t in TEST_TRIALS:
        m = tid == t
        scores_s[m] = smooth(scores[m])

    thresh_final = float(np.mean([r["thresh"] for r in fold_results]))
    preds = (scores_s >= thresh_final).astype(int)
    auc_mean = float(np.mean([r["auc"] for r in fold_results]))
    ap_mean = float(np.mean([r["ap"] for r in fold_results]))

    test_mask = np.isin(tid, TEST_TRIALS)
    evt = event_detection_metrics(
        scores_s[test_mask],
        y_mom_arr[test_mask],
        threshold=thresh_final,
        horizon=horizon,
        fs_eff=FS_EFF,
        refractory=refractory,
    )

    scores_gated = np.zeros_like(scores_s)
    for t in TEST_TRIALS:
        m = tid == t
        scores_gated[m] = (
            persistence_gate(scores_s[m], persist_k) if persist_k > 1 else scores_s[m]
        )

    evt_gated = event_detection_metrics(
        scores_gated[test_mask],
        y_mom_arr[test_mask],
        threshold=thresh_final,
        horizon=horizon,
        fs_eff=FS_EFF,
        refractory=refractory,
    )

    print(f"\nLOTO AUC={auc_mean:.4f}  AP={ap_mean:.4f}  thresh={thresh_final:.4f}")
    print(
        f"No gate : EvtSens={evt['event_sensitivity']:.3f}  "
        f"FA/min={evt['fa_per_minute']:.1f}  "
        f"Latency={evt.get('mean_latency_ms',0):.0f}ms"
    )
    print(
        f"+ gate  : EvtSens={evt_gated['event_sensitivity']:.3f}  "
        f"FA/min={evt_gated['fa_per_minute']:.1f}  "
        f"Latency={evt_gated.get('mean_latency_ms',0):.0f}ms  (persist_k={persist_k})"
    )

    return dict(
        scores=scores_s,
        scores_gated=scores_gated,
        scores_raw=scores,
        auc=auc_mean,
        ap=ap_mean,
        thresh=thresh_final,
        preds=preds,
        evt=evt,
        evt_gated=evt_gated,
        fold_results=fold_results,
    )


print("Utilities ready.")

Utilities ready.


## 8 · Optuna Pipeline Search

Each trial trains one fixed-tree LGB per LOTO fold and scores the result using
`event_sensitivity / (1 + α · FA/min)` on the gated predictions.  
The ratio is bounded `[0, 1]` so TPE's surrogate model always operates on a consistent scale.


In [ ]:
def pipeline_objective(trial):
    T_ms = trial.suggest_int("T_ms", T_MS_MIN, T_MS_MAX, step=50)
    horizon_ms = trial.suggest_int(
        "horizon_ms", HORIZON_MS_MIN, HORIZON_MS_MAX, step=50
    )
    persist_k = trial.suggest_int("persist_k", PERSIST_K_MIN, PERSIST_K_MAX)
    T_samp = max(1, int(T_ms / 1000 * FS_EFF))
    horizon = max(1, int(horizon_ms / 1000 * FS_EFF))
    refractory = max(1, int(REFRACTORY_MS / 1000 * FS_EFF))

    y_labeled = relabel_with_horizon(horizon)

    obj_vals = []
    for fold_i, test_trial in enumerate(TEST_TRIALS):
        te = tid_all == test_trial
        tr = ~te

        X_tr_3d = X_max[tr, -T_samp:, :]
        X_te_3d = X_max[te, -T_samp:, :]

        if USE_EA:
            X_tr_ea, W_fold = euclidean_align(X_tr_3d)
            X_te_ea = (X_te_3d @ W_fold.T).astype(np.float32)
        else:
            X_tr_ea = X_tr_3d.astype(np.float32)
            X_te_ea = X_te_3d.astype(np.float32)

        X_tr_f = features_for_T(T_samp, X_tr_ea)
        X_te_f = features_for_T(T_samp, X_te_ea)

        press_idx = np.where(y_mom_all[tr] > 0)[0]
        template = (
            X_tr_ea[press_idx].mean(0)
            if len(press_idx) > 0
            else np.zeros((T_samp, N_CH), dtype=np.float32)
        )
        X_tr_f = np.concatenate([X_tr_f, mf_features(X_tr_ea, template)], axis=1)
        X_te_f = np.concatenate([X_te_f, mf_features(X_te_ea, template)], axis=1)

        y_tr = y_labeled[tr]
        pp_tr = post_press_mask_all[tr]
        keep = (y_tr == 1) | ~pp_tr
        X_tr_sub, y_tr_sub = subsample(
            X_tr_f[keep],
            y_tr[keep],
            TRAIN_CAP,
            seed=SEED + trial.number * 31 + fold_i,
        )

        model = lgb.LGBMClassifier(
            **FIXED_PARAMS,
            class_weight="balanced",
            random_state=SEED,
            verbose=-1,
            n_jobs=LGB_JOBS,
        )
        model.fit(X_tr_sub, y_tr_sub)

        # Threshold from training-fold ROC
        sc_tr_roc = model.predict_proba(X_tr_sub)[:, 1]
        fpr_t, tpr_t, thr_t = roc_curve(y_tr_sub, sc_tr_roc)
        thresh = thresh_at_spec(fpr_t, tpr_t, thr_t)

        sc_te = model.predict_proba(X_te_f)[:, 1]
        sc_sm = smooth(sc_te)
        sc_g = persistence_gate(sc_sm, persist_k) if persist_k > 1 else sc_sm

        evt = event_detection_metrics(
            sc_g, y_mom_all[te], thresh, horizon, FS_EFF, refractory
        )
        obj = evt["event_sensitivity"] / (1.0 + OBJ_ALPHA * evt["fa_per_minute"])
        obj_vals.append(obj)

        trial.report(float(np.mean(obj_vals)), step=fold_i)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(obj_vals))


study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=SEED),
    pruner=MedianPruner(n_startup_trials=8, n_warmup_steps=3),
)


def _progress(study, trial):
    v = f"{trial.value:.4f}" if trial.value is not None else "---"
    best = f"{study.best_value:.4f}" if study.best_value is not None else "---"
    tag = "PRUNED" if trial.state == optuna.trial.TrialState.PRUNED else f"obj={v}"
    p = study.best_trial.params if study.best_trial else {}
    print(
        f"  trial {trial.number:3d}  {tag}  best={best}  "
        f"T={p.get('T_ms','?'):>4}ms  H={p.get('horizon_ms','?'):>4}ms  "
        f"K={p.get('persist_k','?'):>2}",
        flush=True,
    )


study.optimize(pipeline_objective, n_trials=N_OPTUNA, n_jobs=1, callbacks=[_progress])

BEST = study.best_params
T_OPT = max(1, int(BEST["T_ms"] / 1000 * FS_EFF))
HORIZON = max(1, int(BEST["horizon_ms"] / 1000 * FS_EFF))
PERSIST_K = int(BEST["persist_k"])
REFRACTORY = max(1, int(REFRACTORY_MS / 1000 * FS_EFF))

print(f"\nBest trial #{study.best_trial.number}  obj={study.best_value:.4f}")
print(f"  T        : {BEST['T_ms']} ms  ({T_OPT} samp)")
print(f"  HORIZON  : {BEST['horizon_ms']} ms  ({HORIZON} samp)")
print(f"  PERSIST_K: {PERSIST_K} frames  ({PERSIST_K/FS_EFF*1000:.0f} ms)")
print(f"  REFRAC   : {REFRACTORY_MS} ms  ({REFRACTORY} samp)  [fixed]")

  trial   0  obj=0.1845  best=0.1845  T=1050ms  H=2400ms  K=15
  trial   1  obj=0.0000  best=0.1845  T=1050ms  H=2400ms  K=15
  trial   2  obj=0.2391  best=0.2391  T= 300ms  H=2250ms  K=13
  trial   3  obj=0.0000  best=0.2391  T= 300ms  H=2250ms  K=13


## 8b · Search Diagnostics

In [ ]:
%matplotlib inline

fig, (ax_h, ax_i) = plt.subplots(1, 2, figsize=(14, 4), facecolor=DARK)

# History
completed = [t for t in study.trials if t.value is not None]
obj_hist = [t.value for t in completed]
trial_nos = [t.number for t in completed]
best_curve = np.maximum.accumulate(obj_hist)
ax_h.set_facecolor(CARD)
for sp in ax_h.spines.values():
    sp.set_color(EDGE)
ax_h.tick_params(colors=WHITE, labelsize=8)
ax_h.scatter(trial_nos, obj_hist, color=TEAL, s=18, alpha=0.7, zorder=3, label="trial")
ax_h.plot(trial_nos, best_curve, color=CORAL, lw=1.8, zorder=4, label="best so far")
ax_h.axhline(study.best_value, color=GOLD, lw=0.9, ls="--", alpha=0.7)
ax_h.set_xlabel("Trial", color=WHITE, fontsize=9)
ax_h.set_ylabel(f"sens / (1 + {OBJ_ALPHA}·FA/min)  [0, 1]", color=WHITE, fontsize=9)
ax_h.set_title("Optuna history", color=WHITE, fontsize=10)
ax_h.legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE, fontsize=8)

# Importance
try:
    imp = optuna.importance.get_param_importances(study)
    p_names = list(imp.keys())
    p_vals = [imp[k] for k in p_names]
    ax_i.set_facecolor(CARD)
    for sp in ax_i.spines.values():
        sp.set_color(EDGE)
    ax_i.tick_params(colors=WHITE, labelsize=9)
    ax_i.barh(p_names[::-1], p_vals[::-1], color=LIME, alpha=0.85)
    ax_i.set_xlabel("Importance", color=WHITE, fontsize=9)
    ax_i.set_title("Parameter importance", color=WHITE, fontsize=10)
except Exception as _e:
    ax_i.text(
        0.5,
        0.5,
        f"Importance N/A\n({_e})",
        color=WHITE,
        ha="center",
        va="center",
        transform=ax_i.transAxes,
    )

plt.tight_layout()
os.makedirs("report", exist_ok=True)
plt.savefig(
    "report/pip_optuna_diagnostics.png", dpi=120, bbox_inches="tight", facecolor=DARK
)
plt.show()
print("Saved -> report/pip_optuna_diagnostics.png")

NameError: name 'plt' is not defined

## 9 · Full LOTO Evaluation with Optimal Pipeline

Horizon labels are recomputed with the best HORIZON from Optuna.


In [ ]:
y_all = relabel_with_horizon(HORIZON)
n_pos = int(y_all.sum())
print(
    f"Horizon labels: HORIZON={HORIZON} ({HORIZON/FS_EFF*1000:.0f} ms)  "
    f"pos={n_pos:,} ({n_pos/len(y_all)*100:.1f}%)"
)

fold_importances = []


def lgb_fn(X_tr, y_tr):
    X_sub, y_sub = subsample(X_tr, y_tr, TRAIN_CAP, seed=SEED)
    m = lgb.LGBMClassifier(
        **FIXED_PARAMS,
        class_weight="balanced",
        random_state=SEED,
        verbose=-1,
        n_jobs=-1,
    )
    m.fit(X_sub, y_sub)
    return m


print("Running full LOTO ...")
res = loto_eval(
    lgb_fn,
    X_max,
    T_OPT,
    y_all,
    tid_all,
    y_mom_all,
    on_fit=lambda m: fold_importances.append(m.feature_importances_.copy()),
    horizon=HORIZON,
    persist_k=PERSIST_K,
    refractory=REFRACTORY,
    post_press_mask=post_press_mask_all,
)

scores_s = res["scores"]
thresh = res["thresh"]
preds = res["preds"]
fold_results = res["fold_results"]
evt = res["evt"]
mean_imp = np.mean(fold_importances, axis=0)
std_imp = np.std(fold_importances, axis=0)

Horizon labels: HORIZON=144 (2400 ms)  pos=38,242 (26.4%)
Running full LOTO ...
  fold  0  AUC=0.5686  AP=0.2033  thresh=0.703
  fold  1  AUC=0.6422  AP=0.3925  thresh=0.709
  fold  2  AUC=0.7008  AP=0.3846  thresh=0.698
  fold  3  AUC=0.7257  AP=0.4137  thresh=0.703
  fold  4  AUC=0.7084  AP=0.4145  thresh=0.699
  fold  5  AUC=0.7353  AP=0.4716  thresh=0.700
  fold  6  AUC=0.7451  AP=0.4789  thresh=0.713
  fold  7  AUC=0.7351  AP=0.4360  thresh=0.691
  fold  8  AUC=0.7312  AP=0.4356  thresh=0.713
  fold  9  AUC=0.7120  AP=0.4413  thresh=0.712

LOTO AUC=0.7005  AP=0.4072  thresh=0.7040
No gate : EvtSens=0.760  FA/min=9.2  Latency=1859ms
+ gate  : EvtSens=0.682  FA/min=6.9  Latency=1785ms  (persist_k=14)


## 10 · Save Artifacts

In [ ]:
import pickle

DEMO_TRIAL = TEST_TRIALS[-1]
te_d = tid_all == DEMO_TRIAL
tr_d = ~te_d
X_tr_3d_d = X_max[tr_d, -T_OPT:, :]

if USE_EA:
    X_tr_aligned_d, W_demo = euclidean_align(X_tr_3d_d)
else:
    X_tr_aligned_d = X_tr_3d_d.astype(np.float32)
    W_demo = np.eye(N_CH, dtype=np.float64)

X_feat_d = features_for_T(T_OPT, X_tr_aligned_d)
press_idx_d = np.where(y_mom_all[tr_d] > 0)[0]
template_d = (
    X_tr_aligned_d[press_idx_d].mean(0)
    if len(press_idx_d) > 0
    else np.zeros((T_OPT, N_CH), dtype=np.float32)
)
X_feat_d = np.concatenate([X_feat_d, mf_features(X_tr_aligned_d, template_d)], axis=1)
model_demo = lgb_fn(X_feat_d, y_all[tr_d])

os.makedirs("models", exist_ok=True)
save_path = "models/lgb_pipeline_opt.pkl"
with open(save_path, "wb") as fh:
    pickle.dump(
        dict(
            model=model_demo,
            USE_EA=USE_EA,
            W=W_demo,
            mf_template=template_d,
            T_OPT=T_OPT,
            FS=FS,
            DECIM=DECIM,
            FS_EFF=FS_EFF,
            LP=LP,
            HP=HP,
            HORIZON=HORIZON,
            SMOOTH_WIN=SMOOTH_WIN,
            REFRACTORY=REFRACTORY,
            PERSIST_K=PERSIST_K,
            thresh=thresh,
            EEG_COLS=EEG_COLS,
            optuna_best=BEST,
            optuna_value=study.best_value,
        ),
        fh,
    )
print(f"Saved -> {save_path}")
print(
    f"  T_OPT={T_OPT} ({T_OPT/FS_EFF*1000:.0f}ms)  "
    f"HORIZON={HORIZON} ({HORIZON/FS_EFF*1000:.0f}ms)  "
    f"PERSIST_K={PERSIST_K}  "
    f"REFRACTORY={REFRACTORY} ({REFRACTORY/FS_EFF*1000:.0f}ms)  "
    f"thresh={thresh:.4f}"
)

Saved -> models/lgb_pipeline_opt.pkl
  T_OPT=135 (2250ms)  HORIZON=144 (2400ms)  PERSIST_K=14  REFRACTORY=60 (1000ms)  thresh=0.7040


## 11 · Summary

In [ ]:
e = evt
eg = res["evt_gated"]
print("=" * 64)
print(f"  LightGBM Pipeline Opt -- Subject {SUBJECT}  LOTO CV")
print("=" * 64)
print(f"  Tree params    : fixed from pkl (or defaults)")
print(f"  Optuna trials  : {N_OPTUNA}  objective = sens / (1 + {OBJ_ALPHA}*FA/min)")
print(f"  Best obj value : {study.best_value:.4f}  (range [0, 1])")
print()
print(f"  Window (T_OPT) : {BEST['T_ms']} ms  ({T_OPT} samp @ {FS_EFF} Hz)")
print(f"  Horizon        : {BEST['horizon_ms']} ms  ({HORIZON} samp)")
print(f"  Persist gate   : {PERSIST_K} frames  ({PERSIST_K/FS_EFF*1000:.0f} ms)")
print(f"  Refractory     : {REFRACTORY_MS} ms  ({REFRACTORY} samp)  [fixed]")
print(f"  Smooth win     : {SMOOTH_WIN/FS_EFF*1000:.0f} ms  (fixed)")
print(f"  Threshold      : {thresh:.4f}  (spec >= {MIN_SPEC:.0%}, CV-tuned)")
print()
print(f"  {'Metric':<22} {'No gate':>10}  {'+gate':>10}")
print(f"  {'-'*46}")
print(f"  {'LOTO AUC':<22} {res['auc']:>10.4f}")
print(f"  {'LOTO AP':<22} {res['ap']:>10.4f}")
print(
    f"  {'Event sens.':<22} {e['event_sensitivity']:>10.3f}  {eg['event_sensitivity']:>10.3f}"
)
print(f"  {'FA / min':<22} {e['fa_per_minute']:>10.1f}  {eg['fa_per_minute']:>10.1f}")
print(
    f"  {'Mean latency (ms)':<22} {e.get('mean_latency_ms',0):>10.1f}  {eg.get('mean_latency_ms',0):>10.1f}"
)
print(
    f"  {'Events detected':<22} {e['n_detected']:>5}/{e['n_events']:<4}   {eg['n_detected']:>5}/{eg['n_events']:<4}"
)
print("=" * 64)

  LightGBM Pipeline Opt -- Subject 0  LOTO CV
  Tree params    : fixed from pkl (or defaults)
  Optuna trials  : 50  objective = sens / (1 + 0.1*FA/min)
  Best obj value : 0.4412  (range [0, 1])

  Window (T_OPT) : 2250 ms  (135 samp @ 60 Hz)
  Horizon        : 2400 ms  (144 samp)
  Persist gate   : 14 frames  (233 ms)
  Refractory     : 1000 ms  (60 samp)  [fixed]
  Smooth win     : 100 ms  (fixed)
  Threshold      : 0.7040  (spec >= 95%, CV-tuned)

  Metric                    No gate       +gate
  ----------------------------------------------
  LOTO AUC                   0.7005
  LOTO AP                    0.4072
  Event sens.                 0.760       0.682
  FA / min                      9.2         6.9
  Mean latency (ms)          1858.9      1784.6
  Events detected          203/267      182/267 


## 12 · Trial Probability Timelines

In [ ]:
%matplotlib inline
os.makedirs("report", exist_ok=True)

fold_by_trial = {r["trial"]: r for r in fold_results}
n_rows = (len(ALL_TRIALS) + 1) // 2

fig, axes = plt.subplots(
    n_rows,
    2,
    figsize=(18, 4 * n_rows),
    facecolor=DARK,
    gridspec_kw=dict(hspace=0.45, wspace=0.12),
)
axes_flat = axes.flatten()

for idx, t in enumerate(ALL_TRIALS):
    ax = axes_flat[idx]
    ax.set_facecolor(CARD)
    for sp in ax.spines.values():
        sp.set_color(EDGE)
    ax.tick_params(colors=WHITE, labelsize=7)

    m = tid_all == t
    sc = scores_s[m]
    yl = y_all[m].astype(bool)
    n = m.sum()
    t_sec = (T_MAX + np.arange(n)) / FS_EFF

    ax.fill_between(t_sec, 0, 1, where=yl, color=GOLD, alpha=0.18, step="post")
    ax.plot(t_sec, sc, color=TEAL, lw=0.9, alpha=0.92)
    ax.axhline(thresh, color=CORAL, lw=1.2, ls="--")

    _, y_mom_t = raw_data[t]
    for ps in np.where(y_mom_t > 0)[0]:
        ps_t = ps / FS_EFF
        if t_sec[0] <= ps_t <= t_sec[-1]:
            ax.axvline(ps_t, color=WHITE, lw=0.6, alpha=0.5)

    fd = fold_by_trial.get(t, {})
    title = (
        f"Trial {t}  AUC={fd.get('auc',0):.3f}  AP={fd.get('ap',0):.3f}"
        if t in fold_by_trial
        else f"Trial {t} (train only)"
    )
    ax.set_title(title, color=WHITE, fontsize=9, pad=4)
    ax.set_xlim(t_sec[0], t_sec[-1])
    ax.set_ylim(-0.03, 1.03)
    ax.set_xlabel("Time (s)", color=WHITE, fontsize=7)
    ax.set_ylabel("P(press)", color=WHITE, fontsize=7)

for ax in axes_flat[len(ALL_TRIALS) :]:
    ax.set_visible(False)

handles = [
    mpatches.Patch(color=GOLD, alpha=0.4, label="Horizon label = 1"),
    plt.Line2D([0], [0], color=TEAL, lw=1.5, label="P(press) smoothed"),
    plt.Line2D([0], [0], color=CORAL, lw=1.5, ls="--", label="Threshold"),
    plt.Line2D([0], [0], color=WHITE, lw=1.0, alpha=0.7, label="Button press"),
]
fig.legend(
    handles=handles,
    loc="lower center",
    ncol=4,
    facecolor=CARD,
    labelcolor=WHITE,
    edgecolor=EDGE,
    fontsize=8,
    bbox_to_anchor=(0.5, 0.01),
)
fig.suptitle(
    f"Subject {SUBJECT} -- Pipeline-Opt LOTO  "
    f"AUC={res['auc']:.3f}  "
    f"EvtSens={eg['event_sensitivity']:.3f}  "
    f"FA/min={eg['fa_per_minute']:.1f}  "
    f"T={BEST['T_ms']}ms  H={BEST['horizon_ms']}ms  K={PERSIST_K}  R={REFRACTORY_MS}ms",
    color=WHITE,
    fontsize=11,
    y=0.998,
    fontweight="bold",
)
plt.savefig("report/pip_timelines.png", dpi=120, bbox_inches="tight", facecolor=DARK)
plt.show()
print("Saved -> report/pip_timelines.png")

NameError: name 'os' is not defined